# Plantilla: regresión lineal (agnóstica al dataset)

Notebook **base** para predecir una variable **numérica continua** a partir de un CSV local. No está ligado a un dataset concreto: solo cambias rutas, columnas y, si quieres, la lista en `build_models()` (sklearn + XGBoost + CatBoost).

## Pasos principales (ejecutar en orden)

| Paso | Sección | Qué haces |
|------|---------|-----------|
| **0** | Helpers | Imports y funciones (`infer_feature_columns`, `infer_column_types`, `build_preprocess`, …). |
| **1** | Explorar CSV | `PREVIEW_PATH`, `PREVIEW_SEP` — nombres, tipos, faltantes, sugerencia numéricas/categóricas. |
| **2** | CONFIG | `DATA_PATH`, `TARGET_COL`, `DROP_COLS`; opcional `FEATURE_COLS`, `NUMERIC_COLS`, `CATEGORICAL_COLS`; `build_models()`. |
| **3** | Carga | `pd.read_csv` con los parámetros de CONFIG. |
| **4** | Calidad de datos | Distribución del target y faltantes. |
| **5** | Análisis de features | Histograma, scatter y matriz de correlación (features + target). |
| **6** | Split | **X**, **y**; train / val / test (`split_train_val_test`). |
| **7** | Preprocesado | `ColumnTransformer`: numéricas → imputer + escalar; categóricas → imputer + one-hot. |
| **8** | Comparar modelos | Entrena en **train**; compara train vs **val** (overfitting); orden por **R²** en val. |
| **9** | Mejor modelo | Real vs predicho, correlaciones en test y métricas. | Elige en val; reentrena train+val; evaluación final en **test**. |

Preprocesado dentro del `Pipeline`: `fit` en train, benchmark en **val**, test reservado para el paso 9 (sin *leakage*). Columnas de **X** que no estén en numéricas ni categóricas se descartan (`remainder="drop"`).

### Primera vez con tu CSV

1. Copia el archivo a `data/`.
2. Ejecuta el paso **1** hasta que el `head()` se vea bien (prueba `,`, `;` o `\t` en el separador).
3. Rellena el paso **2** con el mismo path y separador.
4. Ejecuta del paso **3** al **9** sin saltar celdas.

**Ejemplos ya resueltos:** carpeta [`01-regresion/`](01-regresion/) (wine-quality-red, auto-mpg, diabetes)

> Ejecuta Jupyter desde `07.b-ejemplos-supervisados/` para que `data/...` resuelva bien.




In [ ]:
# =============================================================================
# Helpers — funciones reutilizables (misma lógica en todo el benchmark)
# =============================================================================
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer  # pipeline distinto por tipo de columna
from sklearn.impute import SimpleImputer  # rellenar NaN antes de escalar/codificar
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline  # encadena: preprocesado → modelo
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
def infer_feature_columns(df, target_col, drop_cols, feature_cols):
    """Lista de columnas predictoras (X)."""
    if feature_cols is not None:
        return list(feature_cols)
    exclude = {target_col, *drop_cols}
    return [c for c in df.columns if c not in exclude]


def infer_column_types(X, numeric_cols=None, categorical_cols=None):
    """Separa columnas numéricas y categóricas para el ColumnTransformer."""
    if numeric_cols is None:
        numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    if categorical_cols is None:
        categorical_cols = X.select_dtypes(
            include=["object", "category", "bool", "string"]
        ).columns.tolist()
    return list(numeric_cols), list(categorical_cols)


def build_preprocess(numeric_cols, categorical_cols):
    """Preprocesador compartido por todos los modelos (fit solo en train dentro del Pipeline)."""
    transformers = []
    if numeric_cols:
        transformers.append(
            (
                "num",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_cols,
            )
        )
    if categorical_cols:
        transformers.append(
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "encoder",
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        ),
                    ]
                ),
                categorical_cols,
            )
        )
    if not transformers:
        raise ValueError("No hay columnas numéricas ni categóricas para preprocesar.")
    return ColumnTransformer(transformers, remainder="drop")


def split_train_val_test(X, y, test_size, val_size, random_state, stratify=False):
    """Divide en train, validación y test (dos llamadas a train_test_split).

    - test_size: fracción del total para test (hold-out final, paso 9).
    - val_size: fracción de train+val; el benchmark (paso 8) usa solo val.
    Con test_size=0.2 y val_size=0.25 → ~60 % train, ~20 % val, ~20 % test.
    """
    kw = dict(test_size=test_size, random_state=random_state)
    if stratify:
        kw["stratify"] = y
    X_tv, X_test, y_tv, y_test = train_test_split(X, y, **kw)
    kw2 = dict(test_size=val_size, random_state=random_state)
    if stratify:
        kw2["stratify"] = y_tv
    X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, **kw2)
    return X_train, X_val, X_test, y_train, y_val, y_test





## 1. Explorar el CSV (antes de CONFIG)

Pon aquí la ruta de **tu** archivo. No rellenes CONFIG todavía: primero mira nombres de columnas, tipos y nulos.

Si una sola columna contiene todo el CSV, prueba otro `PREVIEW_SEP` (`","`, `";"`, `"\t"`).


In [ ]:
# --- Paso 1: explorar SIN tocar CONFIG todavía ---
PREVIEW_PATH = "data/mi_dataset.csv"  # ruta a tu CSV
PREVIEW_SEP = ","  # separador: ","  |  ";"  |  "\t"

# Carga provisional solo para inspeccionar estructura
df_preview = pd.read_csv(PREVIEW_PATH, sep=PREVIEW_SEP)

print(f"Filas: {len(df_preview):,}  |  Columnas: {len(df_preview.columns)}")
print("\n--- Nombres de columnas (índice : nombre) ---")
for i, col in enumerate(df_preview.columns):
    print(f"  {i:2d}: {col!r}")

print("\n--- Tipos de datos (dtypes) ---")
print(df_preview.dtypes)

print("\n--- Primeras filas ---")
display(df_preview.head())

# Faltantes: el pipeline imputará después; aquí solo diagnosticamos
print("\n--- Valores faltantes por columna ---")
missing = df_preview.isna().sum()
if missing.any():
    display(missing[missing > 0].to_frame("nulos"))
else:
    print("No hay valores faltantes.")

# Ayuda para rellenar NUMERIC_COLS / CATEGORICAL_COLS en CONFIG
_num = df_preview.select_dtypes(include=[np.number]).columns.tolist()
_cat = df_preview.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()
print("\n--- Sugerencia automática de tipos ---")
print("Numéricas (int/float):", _num)
print("Categóricas (object/category/bool/string):", _cat)
print(
    "\n>>> Siguiente: en CONFIG pon DATA_PATH, CSV_SEP iguales y elige TARGET_COL "
    "(columna numérica continua a predecir)."
)



## 2. CONFIG — adaptar a tu dataset

Copia los valores decididos en la exploración. **Solo esta sección** (y opcionalmente `build_models()`) cambia entre proyectos.


In [ ]:
# ========== Paso 2: CONFIG — único bloque que cambia entre datasets ==========
DATA_PATH = "data/mi_dataset.csv"  # mismo path que PREVIEW_PATH
CSV_SEP = ","  # mismo separador que PREVIEW_SEP

TARGET_COL = "nombre_columna_objetivo"  # variable numérica continua (precio, mpg, quality…)

# Columnas que no deben usarse como features (ids, texto libre, leakage)
DROP_COLS = []  # ej. ["id", "car_name"]

# None = automático; o listas explícitas si la inferencia falla
FEATURE_COLS = None
NUMERIC_COLS = None
CATEGORICAL_COLS = None

TEST_SIZE = 0.2  # fracción total para test (hold-out final)
VAL_SIZE = 0.25  # fracción de train+val → validación (~20 % del total si TEST_SIZE=0.2)
RANDOM_STATE = 42  # reproducibilidad del split y modelos
METRIC_PRINCIPAL = "r2"  # columna para ordenar la tabla (mayor = mejor en regresión)


def build_models():
    """Diccionario nombre → estimador. Comenta líneas para excluir modelos del benchmark."""
    from sklearn.ensemble import (
        GradientBoostingRegressor,
        HistGradientBoostingRegressor,
        RandomForestRegressor,
    )
    from sklearn.linear_model import Lasso, LinearRegression, Ridge
    from sklearn.neighbors import KNeighborsRegressor
    from xgboost import XGBRegressor
    from catboost import CatBoostRegressor

    models = {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(random_state=RANDOM_STATE),
        "Lasso": Lasso(random_state=RANDOM_STATE, max_iter=5000),
        "RandomForest": RandomForestRegressor(
            n_estimators=100,
            criterion="squared_error",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features=1.0,
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            bootstrap=True,
            oob_score=False,
            max_samples=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
        "HistGradientBoosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
        "KNN": KNeighborsRegressor(n_neighbors=5, n_jobs=-1),
        "XGBoost": XGBRegressor(
            random_state=RANDOM_STATE, verbosity=0, n_estimators=100, n_jobs=-1
        ),
        "CatBoost": CatBoostRegressor(
            random_state=RANDOM_STATE,
            verbose=False,
            iterations=100,
            allow_writing_files=False,
        ),
    }
    return models


MODELS = build_models()



## 3. Carga de datos

Vuelve a leer el CSV con los parámetros ya fijados en CONFIG.


In [ ]:
# --- Paso 3: carga definitiva con los parámetros de CONFIG ---
df = pd.read_csv(DATA_PATH, sep=CSV_SEP)
print("Shape:", df.shape)
df.head()



## 4. Calidad de datos

Diagnóstico de faltantes (el relleno real lo hace el pipeline en train).


In [ ]:
# --- Paso 4: calidad de datos (diagnóstico; el imputer actúa en el Pipeline) ---
print(df.info())
print("\nFaltantes por columna:")
missing = df.isna().sum()
print(missing[missing > 0] if missing.any() else "Sin valores faltantes")



## 5. Análisis de features y dependencias

Explora la relación del **target** con las predictoras: histograma, scatter y **matriz de correlación** (solo tiene sentido en regresión; la matriz de confusión es para clasificación).



In [ ]:
# --- Paso 5: análisis de features y dependencias con el target ---
feature_cols_eda = infer_feature_columns(df, TARGET_COL, DROP_COLS, FEATURE_COLS)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df[TARGET_COL].hist(ax=axes[0], bins=20, edgecolor="black")
axes[0].set_title(f"Distribución de {TARGET_COL}")

num_feat = df[feature_cols_eda].select_dtypes(include=[np.number]).columns
if len(num_feat):
    corrs = df[num_feat].corrwith(df[TARGET_COL]).abs().sort_values(ascending=False)
    feat = corrs.index[0] if len(corrs.dropna()) else num_feat[0]
    axes[1].scatter(df[feat], df[TARGET_COL], alpha=0.4)
    axes[1].set_xlabel(feat)
    axes[1].set_ylabel(TARGET_COL)

plt.tight_layout()
plt.show()

# --- Correlación entre features numéricas y target ---
num_cols = [
    c for c in feature_cols_eda
    if c in df.columns and pd.api.types.is_numeric_dtype(df[c])
]
cols_corr = list(dict.fromkeys(num_cols + [TARGET_COL]))

top_feat = None
if len(cols_corr) >= 2 and pd.api.types.is_numeric_dtype(df[TARGET_COL]):
    corr = df[cols_corr].corr(numeric_only=True)
    size = max(5, 0.75 * len(cols_corr))
    fig, ax = plt.subplots(figsize=(size, size))
    sns.heatmap(
        corr,
        annot=len(cols_corr) <= 12,
        fmt=".2f",
        cmap="RdBu_r",
        center=0,
        vmin=-1,
        vmax=1,
        ax=ax,
    )
    ax.set_title("Correlación: features numéricas y target")
    plt.tight_layout()
    plt.show()
    s = corr[TARGET_COL].drop(TARGET_COL, errors="ignore").abs()
    if len(s):
        top_feat = s.idxmax()

if top_feat is None:
    others = [c for c in feature_cols_eda if c in df.columns and c != TARGET_COL]
    top_feat = others[0] if others else None



## 6. Separar X / y y split train-test

En regresión no hace falta `stratify` (solo en clasificación).


In [ ]:
# --- Paso 6: separar features (X), target (y) y dividir train / test ---
feature_cols = infer_feature_columns(df, TARGET_COL, DROP_COLS, FEATURE_COLS)
X = df[feature_cols]
y = df[TARGET_COL]

# Clasificación de columnas para el ColumnTransformer
numeric_cols, categorical_cols = infer_column_types(X, NUMERIC_COLS, CATEGORICAL_COLS)

print("Numéricas:", numeric_cols)
print("Categóricas:", categorical_cols)


X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_STATE
)
print(
    f"Tamaños → train: {len(X_train):,} | val: {len(X_val):,} | test: {len(X_test):,}"
)



## 7. Preprocesado (compartido por todos los modelos)

Un solo `ColumnTransformer` para todos los modelos del benchmark.


In [ ]:
# --- Paso 7: definir el preprocesador (mismo objeto para todos los modelos) ---
preprocess = build_preprocess(numeric_cols, categorical_cols)
preprocess  # muestra la estructura: ramas num y cat



## 8. Comparar modelos

Métricas en **train** y **val**; brecha train−val para detectar **overfitting**.

Entrena en **train** y evalúa en **val** cada entrada de `MODELS` con el mismo split train/val y el mismo preprocesado. Puede tardar varios minutos.



In [ ]:
# --- Paso 8: métricas y benchmark de modelos ---

OVERFIT_GAP_WARN = 0.15  # brecha train-val en METRIC_PRINCIPAL por encima → posible overfitting
# R²: gap = train - val (mayor = peor generalización). MAE/RMSE: gap = val - train.

def regression_metrics(y_true, y_pred):
    """MAE, RMSE y R²."""
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    return {"mae": mae, "rmse": rmse, "r2": r2}


def _generalization_gap(m_train, m_val, metric):
    if metric == "r2":
        return m_train[metric] - m_val[metric]
    return m_val[metric] - m_train[metric]


def evaluate_models(models, preprocess, X_train, X_val, y_train, y_val):
    """Entrena en train; métricas en train y val para detectar overfitting."""
    rows = []
    metric = METRIC_PRINCIPAL
    for name, estimator in models.items():
        pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
        pipe.fit(X_train, y_train)
        m_train = regression_metrics(y_train, pipe.predict(X_train))
        m_val = regression_metrics(y_val, pipe.predict(X_val))
        row = {"modelo": name}
        for k, v in m_train.items():
            row[f"{k}_train"] = v
        for k, v in m_val.items():
            row[f"{k}_val"] = v
        row[f"gap_{metric}"] = _generalization_gap(m_train, m_val, metric)
        rows.append(row)
    asc = metric not in ("r2",)
    return pd.DataFrame(rows).sort_values(f"{metric}_val", ascending=asc)


results = evaluate_models(MODELS, preprocess, X_train, X_val, y_train, y_val)
display(results.round(4))

gap_col = f"gap_{METRIC_PRINCIPAL}"
sospechosos = results[results[gap_col] > OVERFIT_GAP_WARN]
if len(sospechosos):
    print(
        f"\nPosible overfitting (gap {METRIC_PRINCIPAL} train-val > {OVERFIT_GAP_WARN}):"
    )
    display(
        sospechosos[
            ["modelo", f"{METRIC_PRINCIPAL}_train", f"{METRIC_PRINCIPAL}_val", gap_col]
        ].round(4)
    )
else:
    print(
        f"\nSin gap {METRIC_PRINCIPAL} train-val > {OVERFIT_GAP_WARN} "
        "(no hay señal fuerte de overfitting en el benchmark)."
    )

fig, ax = plt.subplots(figsize=(9, 5))
plot_df = results.set_index("modelo")[
    [f"{METRIC_PRINCIPAL}_train", f"{METRIC_PRINCIPAL}_val"]
]
plot_df.plot(kind="barh", ax=ax)
ax.set_xlabel(METRIC_PRINCIPAL)
ax.set_title("Train vs validación — regresión")
ax.legend(["train", "val"])
plt.tight_layout()
plt.show()





## 9. Detalle del mejor modelo

Scatter **real vs predicho**, **matriz de correlación** en test (target, predicción y features numéricas) y métricas (reentreno train+val).


In [ ]:
# --- Paso 9: detalle del mejor modelo según METRIC_PRINCIPAL ---
best_name = results.iloc[0]["modelo"]
print(f"Mejor modelo (val): {best_name}")

best_est = MODELS[best_name]
X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

best_pipe = Pipeline([("preprocess", preprocess), ("model", best_est)])
best_pipe.fit(X_trainval, y_trainval)
y_pred_test = best_pipe.predict(X_test)

print("Métricas en test (tras reentrenar con train+val):")
display(pd.DataFrame([regression_metrics(y_test, y_pred_test)]).round(4))

# Scatter: cada punto es una fila de test (eje x = real, eje y = predicho)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred_test, alpha=0.5, edgecolors="k", linewidths=0.3)
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
ax.plot(lims, lims, "r--", lw=1)  # línea ideal y=x
ax.set_xlabel("Valor real")
ax.set_ylabel("Predicción")
ax.set_title(f"{best_name}: real vs predicho (test)")
plt.tight_layout()
plt.show()

# Matriz de correlación en test: target, predicción y features numéricas
_num_test = X_test.select_dtypes(include=[np.number]).columns.tolist()
if len(_num_test) >= 1:
    corr_test = pd.concat(
        [
            X_test[_num_test],
            pd.Series(y_test.values, name=TARGET_COL, index=X_test.index),
            pd.Series(y_pred_test, name="prediccion", index=X_test.index),
        ],
        axis=1,
    ).corr(numeric_only=True)
    size = max(5, 0.75 * len(corr_test.columns))
    fig, ax = plt.subplots(figsize=(size, size))
    sns.heatmap(
        corr_test,
        annot=len(corr_test.columns) <= 12,
        fmt=".2f",
        cmap="RdBu_r",
        center=0,
        vmin=-1,
        vmax=1,
        ax=ax,
    )
    ax.set_title(f"Correlación en test — {best_name}")
    plt.tight_layout()
    plt.show()
else:
    print("No hay columnas numéricas en X_test para la matriz de correlación.")






## Checklist: nuevo dataset

1. CSV en `data/` → explorar (sección 1) → CONFIG (sección 2).
2. `TARGET_COL` debe ser **numérica continua** (precio, cantidad, puntuación…).
3. `DROP_COLS`: ids, texto libre, columnas que no deben influir en la predicción.
4. Si la detección automática de tipos falla, define `NUMERIC_COLS` / `CATEGORICAL_COLS`.
5. Opcional: comenta modelos en `build_models()` para acortar tiempo.
6. Ejecuta todas las celdas y compara la tabla (R², MAE, RMSE).
